# Первичный аудит датасета Student Performance

Цель: проверить качество данных, пропуски, дубликаты, константные признаки, распределения, баланс целевых интервалов и выбросы. Источник CSV: [Kaggle — Student Performance Multiple Linear Regression](https://www.kaggle.com/datasets/nikhil7280/student-performance-multiple-linear-regression). Скачайте `Student_Performance.csv` и положите рядом с ноутбуком.

В исходном описании проекта заявлены 10 000 строк и 6 столбцов. Ниже эти характеристики проверяются по фактическому CSV, а не принимаются на веру.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('Student_Performance.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Поместите Student_Performance.csv рядом с ноутбуком. Файл можно скачать со страницы Kaggle, указанной выше.')
df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()
print(f'Размер исходной таблицы: {df.shape[0]} строк × {df.shape[1]} столбцов')
display(df.head())


## 1. Общая статистика и профиль таблицы

In [ ]:
profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'n_unique': df.nunique(dropna=True),
    'unique_pct': (df.nunique(dropna=True) / max(len(df), 1) * 100).round(2),
})
display(profile)
display(df.describe(include='all').T)
print('Дубликаты строк:', int(df.duplicated().sum()))
print('Полностью пустые строки:', int(df.isna().all(axis=1).sum()))


## 2. Пропуски: оценка и заполнение

In [ ]:
missing_before = df.isna().sum().sort_values(ascending=False)
display(missing_before[missing_before > 0].rename('missing_count').to_frame())

# Числовые признаки заполняем медианой, категориальные — модой.
# Если пропусков нет, таблица не изменится.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
for col in numeric_cols:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    if df[col].isna().any():
        mode = df[col].mode(dropna=True)
        df[col] = df[col].fillna(mode.iloc[0] if not mode.empty else 'Unknown')
print('Пропусков до заполнения:', int(missing_before.sum()))
print('Пропусков после заполнения:', int(df.isna().sum().sum()))


## 3. Дубликаты: обнаружение и удаление

In [ ]:
duplicates_before = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(f'Удалено дубликатов: {duplicates_before}')
print(f'Размер после удаления дубликатов: {df.shape[0]} × {df.shape[1]}')


## 4. Константные и почти константные признаки

In [ ]:
nunique = df.nunique(dropna=False)
constant_cols = nunique[nunique <= 1].index.tolist()
print('Константные признаки:', constant_cols if constant_cols else 'не обнаружены')
near_constant = pd.DataFrame({
    'most_frequent_share_pct': (df.apply(lambda s: s.value_counts(normalize=True, dropna=False).max()) * 100).round(2),
    'n_unique': df.nunique(dropna=False)
}).sort_values('most_frequent_share_pct', ascending=False)
display(near_constant)


## 5. Баланс целевой переменной

`Performance Index` — числовая целевая переменная для задачи регрессии, поэтому классического баланса классов у неё нет. Для диагностической оценки распределения дополнительно разобьём её на интервалы по квантилям и покажем распределение исходных значений. Интервалы нужны только для аудита, не для замены исходной цели.

In [ ]:
target = 'Performance Index'
if target not in df.columns:
    print(f'Целевая колонка {target!r} не найдена. Проверьте названия столбцов:', df.columns.tolist())
else:
    display(df[target].describe().to_frame('Performance Index'))
    plt.figure(figsize=(9, 4))
    sns.histplot(df[target], bins=30, kde=True)
    plt.title('Распределение Performance Index')
    plt.tight_layout(); plt.show()
    try:
        bands = pd.qcut(df[target], q=5, duplicates='drop')
        balance = bands.value_counts(sort=False).rename_axis('интервал').to_frame('количество')
        balance['доля_%'] = (balance['количество'] / len(df) * 100).round(2)
        display(balance)
        balance['количество'].plot(kind='bar', figsize=(9, 4), title='Частоты квантильных интервалов Performance Index')
        plt.ylabel('Количество записей'); plt.tight_layout(); plt.show()
    except ValueError as e:
        print('Не удалось построить квантильные интервалы:', e)


## 6. Распределения числовых признаков

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T)
for col in numeric_cols:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    sns.histplot(df[col], kde=True, ax=axes[0])
    axes[0].set_title(f'Гистограмма: {col}')
    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f'Boxplot: {col}')
    plt.tight_layout(); plt.show()


## 7. Boxplot приоритетных признаков и проверка выбросов

Для этого датасета приоритетны `Hours Studied`, `Previous Scores`, `Sleep Hours`, `Sample Question Papers Practiced` и целевой `Performance Index`. Границы IQR — диагностический флаг, а не доказательство ошибки: крайние наблюдения нельзя удалять без предметного обоснования.

In [ ]:
priority_cols = [c for c in ['Hours Studied', 'Previous Scores', 'Sleep Hours', 'Sample Question Papers Practiced', 'Performance Index'] if c in df.columns]
outlier_rows = []
for col in priority_cols:
    s = pd.to_numeric(df[col], errors='coerce').dropna()
    q1, q3 = s.quantile([.25, .75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < low) | (s > high)
    outlier_rows.append({'признак': col, 'Q1': q1, 'Q3': q3, 'IQR': iqr,
                         'нижняя_граница': low, 'верхняя_граница': high,
                         'число_выбросов_IQR': int(mask.sum()),
                         'доля_выбросов_%': round(mask.mean()*100, 2) if len(s) else 0})
    plt.figure(figsize=(9, 2.5))
    sns.boxplot(x=s)
    plt.title(f'Boxplot — {col}')
    plt.tight_layout(); plt.show()
outlier_summary = pd.DataFrame(outlier_rows)
display(outlier_summary)


## 8. Проверка допустимых значений и итоговый профиль

In [ ]:
# Проверяем известные диапазоны/категории; это эвристики, их нужно сверить с документацией источника.
if 'Extracurricular Activities' in df.columns:
    print('Категории Extracurricular Activities:', sorted(df['Extracurricular Activities'].dropna().astype(str).unique().tolist()))
for col in ['Hours Studied', 'Sleep Hours', 'Sample Question Papers Practiced']:
    if col in df.columns:
        print(f'{col}: min={df[col].min()}, max={df[col].max()}, отрицательных={(pd.to_numeric(df[col], errors="coerce") < 0).sum()}')

final_profile = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'non_null': df.notna().sum(),
    'missing': df.isna().sum(),
    'unique': df.nunique(dropna=False),
    'example': [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
})
display(final_profile)
print(f'Итоговый размер очищенной таблицы: {df.shape[0]} строк × {df.shape[1]} столбцов')
print('Пропусков:', int(df.isna().sum().sum()))
print('Дубликатов:', int(df.duplicated().sum()))
print('Константных столбцов:', [c for c in df.columns if df[c].nunique(dropna=False) <= 1])

# Сохраняем обработанную таблицу и профиль для отчёта.
df.to_csv('Student_Performance_cleaned.csv', index=False)
final_profile.to_csv('data_profile.csv', index=True)
outlier_summary.to_csv('outlier_summary.csv', index=False)


## 9. Выводы (заполнить после выполнения ноутбука)

По результатам выполнения зафиксируйте фактические значения: размер до/после очистки, число пропусков и дубликатов, наличие константных признаков, частоты квантильных интервалов и количество IQR-выбросов. Не следует заранее утверждать, что пропусков или выбросов нет — это зависит от проверенного CSV.

Важно: `Performance Index` является числовой целью регрессии. Квантильные интервалы приведены только как вспомогательная оценка распределения, а не как исходные классы. Выбросы по IQR следует изучить, но не удалять автоматически без обоснования.